# ACT Push-T: resume the September 17 L4 run
Run this notebook only after the original runtime has stopped.
40,000 steps; all 206 demonstrations; batch 64; 2 workers.
GitHub token is read from Colab Secrets, never embedded.


In [ ]:
import os, sys, subprocess, pathlib, json, shutil, time
import torch
from google.colab import drive, userdata
assert torch.cuda.is_available()
assert any(name in torch.cuda.get_device_name(0) for name in ('L4','T4'))
print(torch.cuda.get_device_name(0))
drive.mount('/content/drive')
os.environ['GH_TOKEN']=userdata.get('GH_TOKEN')
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','gh','ffmpeg'],check=True)
subprocess.run(['gh','auth','setup-git'],check=True)
repo=pathlib.Path('/content/act-pusht')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/imwaterhuang/act-pusht.git',str(repo)],check=True)
os.chdir(repo)
# Preserve the exact model/training revision used by this run.
subprocess.run(['git','checkout','53dddff105b664612bedcf150cbc1f1fd2542bb6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.','--no-deps'],check=True)
subprocess.run([sys.executable,'scripts/download_dataset.py'],check=True)
print('SETUP_READY')


In [ ]:
run=pathlib.Path('/content/act-work/act-seed0-20260917-l4')
mirror=pathlib.Path('/content/drive/MyDrive/act-pusht-runs/act-seed0-20260917-l4')
pidfile=pathlib.Path('/content/act-formal.pid')
if pidfile.exists():
    previous=int(pidfile.read_text())
    command=pathlib.Path(f'/proc/{previous}/cmdline')
    if command.exists() and b'train_act.py' in command.read_bytes():
        raise RuntimeError('Training is already running in this runtime; do not start a duplicate.')
assert (mirror/'checkpoints/last.pt').is_file(), 'No recovery checkpoint'
if (mirror/'completed.json').exists():
    raise RuntimeError('This training run is already complete.')
# Restore into a fresh local directory, never train directly on Drive.
if run.exists():
    run=run.with_name(run.name+'-resume-'+time.strftime('%Y%m%d-%H%M%S'))
shutil.copytree(mirror,run)
env=os.environ.copy();env.pop('GH_TOKEN',None)
env.update(CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='4',MKL_NUM_THREADS='4')
launch='import torch,runpy; torch.set_num_threads(4); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True; torch.use_deterministic_algorithms(True); runpy.run_path("scripts/train_act.py",run_name="__main__")'
log=pathlib.Path('/content/act-formal.log')
handle=log.open('a')
process=subprocess.Popen([sys.executable,'-u','-c',launch,'--config',str(run/'config.yaml'),'--resume',str(run/'checkpoints/last.pt'),'--mirror-dir',str(mirror),'--device','cuda','--num-workers','2'],stdout=handle,stderr=subprocess.STDOUT,env=env)
pidfile.write_text(str(process.pid))
print('RESUMED',process.pid)


In [ ]:
while process.poll() is None:
    lines=log.read_text().splitlines()
    print(time.strftime('%H:%M:%S'), lines[-1] if lines else 'starting', flush=True)
    time.sleep(30)
print('EXIT',process.returncode)
print(log.read_text()[-2000:])
assert process.returncode==0
assert (mirror/'completed.json').exists()
print('ACT_TRAINING_COMPLETE')
